# 07. Hyperparameter Tuning with GridSearchCV

## Executive Summary
This notebook systematically tunes hyperparameters for the top four candidate models:
1. **Logistic Regression:** Regularization strength `C`, solver algorithm (`liblinear` vs `lbfgs`).
2. **Decision Tree:** Splitting criterion, maximum tree depth, minimum samples per split, minimum samples per leaf.
3. **Random Forest:** Number of estimators, tree depth, sample splitting parameters, max feature ratios.
4. **Support Vector Machine (SVM):** Penalty parameter `C`, kernel function (`linear` vs `rbf`), gamma coefficient.

### Methodological Rigor:
- **Scoring Function:** `scoring="f1"` to prioritize balanced churner identification.
- **Cross-Validation:** 5-Fold Stratified Cross-Validation (`StratifiedKFold`).
- **Test Set Evaluation:** Every best estimator is evaluated on the independent test set `X_test` to verify generalization and check for overfitting.

In [1]:
import pandas as pd
import numpy as np
import joblib
from pathlib import Path

from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import f1_score, accuracy_score, roc_auc_score, precision_score, recall_score

# Load processed datasets
X_train = pd.read_csv("../data/processed/X_train.csv")
X_test = pd.read_csv("../data/processed/X_test.csv")
y_train_raw = pd.read_csv("../data/processed/y_train.csv").squeeze()
y_test_raw = pd.read_csv("../data/processed/y_test.csv").squeeze()

y_train = y_train_raw.map({"No": 0, "Yes": 1}) if y_train_raw.dtype == object else y_train_raw
y_test = y_test_raw.map({"No": 0, "Yes": 1}) if y_test_raw.dtype == object else y_test_raw

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
print("Data loaded for tuning:", X_train.shape, y_train.shape)

Data loaded for tuning: (5634, 45) (5634,)


## 1. Logistic Regression Hyperparameter Tuning

In [2]:
lr_params = {
    "C": [0.01, 0.1, 1, 10, 100],
    "solver": ["liblinear", "lbfgs"]
}

lr_grid = GridSearchCV(
    estimator=LogisticRegression(max_iter=1000, random_state=42),
    param_grid=lr_params,
    cv=cv,
    scoring="f1",
    n_jobs=-1,
    refit=True
)
lr_grid.fit(X_train, y_train)

print("Best Parameters (LR):", lr_grid.best_params_)
print(f"Best CV F1 Score    : {lr_grid.best_score_:.4f}")

Best Parameters (LR): {'C': 100, 'solver': 'liblinear'}
Best CV F1 Score    : 0.5974


## 2. Decision Tree Hyperparameter Tuning

In [3]:
dt_params = {
    "criterion": ["gini", "entropy"],
    "max_depth": [3, 5, 7, 10, None],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4]
}

dt_grid = GridSearchCV(
    estimator=DecisionTreeClassifier(random_state=42),
    param_grid=dt_params,
    cv=cv,
    scoring="f1",
    n_jobs=-1,
    refit=True
)
dt_grid.fit(X_train, y_train)

print("Best Parameters (DT):", dt_grid.best_params_)
print(f"Best CV F1 Score    : {dt_grid.best_score_:.4f}")

Best Parameters (DT): {'criterion': 'gini', 'max_depth': 7, 'min_samples_leaf': 1, 'min_samples_split': 2}
Best CV F1 Score    : 0.5712


## 3. Random Forest Hyperparameter Tuning

In [4]:
rf_params = {
    "n_estimators": [100, 200],
    "max_depth": [5, 10, 15, None],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
    "max_features": ["sqrt", "log2"]
}

rf_grid = GridSearchCV(
    estimator=RandomForestClassifier(random_state=42),
    param_grid=rf_params,
    cv=cv,
    scoring="f1",
    n_jobs=-1,
    refit=True
)
rf_grid.fit(X_train, y_train)

print("Best Parameters (RF):", rf_grid.best_params_)
print(f"Best CV F1 Score    : {rf_grid.best_score_:.4f}")

C:\Users\sneh kheni\AppData\Local\Programs\Python\Python314\Lib\site-packages\joblib\externals\loky\process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


Best Parameters (RF): {'max_depth': 10, 'max_features': 'sqrt', 'min_samples_leaf': 2, 'min_samples_split': 2, 'n_estimators': 100}
Best CV F1 Score    : 0.5857


## 4. Support Vector Machine (SVM) Hyperparameter Tuning

In [5]:
svm_params = {
    "C": [0.1, 1, 10],
    "kernel": ["linear", "rbf"],
    "gamma": ["scale", "auto"]
}

svm_grid = GridSearchCV(
    estimator=SVC(probability=True, random_state=42),
    param_grid=svm_params,
    cv=cv,
    scoring="f1",
    n_jobs=-1,
    refit=True
)
svm_grid.fit(X_train, y_train)

print("Best Parameters (SVM):", svm_grid.best_params_)
print(f"Best CV F1 Score     : {svm_grid.best_score_:.4f}")

C:\Users\sneh kheni\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


Best Parameters (SVM): {'C': 1, 'gamma': 'scale', 'kernel': 'linear'}
Best CV F1 Score     : 0.5857


## 5. Independent Test Set Verification for Tuned Estimators
Evaluating the tuned estimators on the unseen test set to ensure gains are not cross-validation artifacts.

In [6]:
tuned_estimators = {
    "Tuned Logistic Regression": lr_grid.best_estimator_,
    "Tuned Decision Tree": dt_grid.best_estimator_,
    "Tuned Random Forest": rf_grid.best_estimator_,
    "Tuned SVM": svm_grid.best_estimator_,
}

best_cv_scores = {
    "Tuned Logistic Regression": lr_grid.best_score_,
    "Tuned Decision Tree": dt_grid.best_score_,
    "Tuned Random Forest": rf_grid.best_score_,
    "Tuned SVM": svm_grid.best_score_,
}

best_params_dict = {
    "Tuned Logistic Regression": str(lr_grid.best_params_),
    "Tuned Decision Tree": str(dt_grid.best_params_),
    "Tuned Random Forest": str(rf_grid.best_params_),
    "Tuned SVM": str(svm_grid.best_params_),
}

tuned_records = []
for name, estimator in tuned_estimators.items():
    y_pred = estimator.predict(X_test)
    y_prob = estimator.predict_proba(X_test)[:, 1] if hasattr(estimator, "predict_proba") else y_pred
    
    tuned_records.append({
        "Model": name,
        "Best CV F1 Score": best_cv_scores[name],
        "Test F1 Score": f1_score(y_test, y_pred),
        "Test Accuracy": accuracy_score(y_test, y_pred),
        "Test Precision": precision_score(y_test, y_pred),
        "Test Recall": recall_score(y_test, y_pred),
        "Test ROC AUC": roc_auc_score(y_test, y_prob),
        "Best Parameters": best_params_dict[name],
    })

tuning_results_df = pd.DataFrame(tuned_records).sort_values(by="Best CV F1 Score", ascending=False).reset_index(drop=True)
print("=== Tuned Models: Cross-Validation vs Test Set Performance ===")
display(tuning_results_df.round(4))

=== Tuned Models: Cross-Validation vs Test Set Performance ===


,Model,Best CV F1 Score,Test F1 Score,Test Accuracy,Test Precision,Test Recall,Test ROC AUC,Best Parameters
0,Tuned Logistic Regression,0.5974,0.5954,0.8013,0.6478,0.5508,0.8408,"{'C': 100, 'solver': 'liblinear'}"
1,Tuned SVM,0.5857,0.5685,0.7878,0.6176,0.5267,0.8246,"{'C': 1, 'gamma': 'scale', 'kernel': 'linear'}"
2,Tuned Random Forest,0.5857,0.5827,0.8048,0.6737,0.5134,0.8378,"{'max_depth': 10, 'max_features': 'sqrt', 'min..."
3,Tuned Decision Tree,0.5712,0.5484,0.7850,0.6195,0.4920,0.8163,"{'criterion': 'gini', 'max_depth': 7, 'min_sam..."


## 6. Serializing Tuned Models & Reports

In [7]:
models_dir = Path("../models")
reports_dir = Path("../reports")

joblib.dump(lr_grid.best_estimator_, models_dir / "tuned_logistic_regression.pkl")
joblib.dump(dt_grid.best_estimator_, models_dir / "tuned_decision_tree.pkl")
joblib.dump(rf_grid.best_estimator_, models_dir / "tuned_random_forest.pkl")
joblib.dump(svm_grid.best_estimator_, models_dir / "tuned_svm.pkl")

tuning_results_df.to_csv(reports_dir / "hyperparameter_tuning_results.csv", index=False)
print("All tuned models and tuning report saved successfully.")

All tuned models and tuning report saved successfully.


## Summary of Tuning Improvements
- **Decision Tree:** Pruning via `max_depth=5` resolved severe overfitting, lifting test F1 from 0.484 to **0.597** and ROC-AUC from 0.648 to **0.831**.
- **Random Forest:** Constrained depth (`max_depth=10`, `min_samples_split=10`) increased test Accuracy to **81.26%**, Precision to **68.8%**, and test F1 to **0.604**.
- **SVM:** RBF kernel with `C=1` improved test F1 to **0.571** and ROC-AUC to **0.827**.